In [1]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/22 21:57:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/22 21:57:34 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/22 21:57:34 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/22 21:57:34 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/22 21:57:34 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/08/22 21:57:34 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/22 21:57:34 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
(sedona
    .read
    .format("binaryFile")
    .load(f"s3a://{bucket_name}/source_data/fdi_data")
    .selectExpr("RS_FromGeoTiff(content) AS rast")
    .createOrReplaceTempView("ffdi"))

(
    sedona
        .read
        .format("binaryFile")
        .load(f"s3a://{bucket_name}/source_data/world_population_raster")
        .selectExpr("RS_FromGeoTiff(content) AS rast")
        .createOrReplaceTempView("population")
)

In [4]:
sedona.sql(
    """
    SELECT 
        raster.tile as rast,
        raster.x,
        raster.y 
    FROM ffdi
    LATERAL VIEW RS_TileExplode(rast, 100, 100) raster
    """
).createOrReplaceTempView("fdi_tiles")

In [5]:
geometry = sedona.sql(
    """
    WITH pixelized AS (
        SELECT 
            RS_PixelAsPolygons(rast, 1) AS pixels,
            x,
            y
        FROM fdi_tiles
    ),
    classified AS (
        SELECT
            pixel.geom,
            x,
            y,
            CASE
                WHEN pixel.value > 50 THEN 'extreme'
                WHEN pixel.value > 25 THEN 'very high'
                WHEN pixel.value > 12 THEN 'high'
                WHEN pixel.value > 5 THEN 'moderate'
                WHEN pixel.value > 0 THEN 'low'
            END AS fire_danger_class
        FROM pixelized
        LATERAL VIEW explode(pixels) AS pixel
        WHERE pixel.value > 0 AND pixel.value < 255
    )
     SELECT
            ST_Union_Aggr(geom) AS geom,
            fire_danger_class
        FROM classified
        GROUP BY fire_danger_class, x, y
    """
).createOrReplaceTempView("fire_danger")

In [6]:
sedona.sql("SELECT * FROM fire_danger").show(5)

[Stage 3:>                                                          (0 + 1) / 1]

+--------------------+-----------------+
|                geom|fire_danger_class|
+--------------------+-----------------+
|MULTIPOLYGON (((-...|             high|
|MULTIPOLYGON (((7...|              low|
|MULTIPOLYGON (((3...|         moderate|
|MULTIPOLYGON (((7...|         moderate|
|MULTIPOLYGON (((-...|             high|
+--------------------+-----------------+
only showing top 5 rows



In [7]:
## SELECT  RS_ZonalStats(rast, 1, ST_CollectionExtrac(geom), 1, 'sum', true, false) AS population_sum
sedona.sql(
    
    """
    WITH intersection AS (
        SELECT 
            rast,
            ST_Buffer(ST_Intersection(RS_Envelope(rast), geom), -0.0001) AS geom,
            fire_danger_class
        FROM population AS p
        JOIN fire_danger AS f ON RS_Intersects(p.rast, f.geom)
    ),
    zonal_stats AS (
        SELECT 
            RS_ZonalStats(rast, geom, 1, 'sum') AS population_sum,
            fire_danger_class
        FROM intersection
    )
    SELECT 
        fire_danger_class,
        CAST(sum(population_sum) AS DECIMAL(38, 0)) AS population_sum
    FROM zonal_stats
    GROUP BY fire_danger_class

    """
).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[fire_danger_class#55], functions=[sum(population_sum#108)])
   +- Exchange hashpartitioning(fire_danger_class#55, 200), ENSURE_REQUIREMENTS, [plan_id=203]
      +- HashAggregate(keys=[fire_danger_class#55], functions=[partial_sum(population_sum#108)])
         +- Project [ **org.apache.spark.sql.sedona_sql.expressions.raster.RS_ZonalStats**   AS population_sum#108, fire_danger_class#55]
            +- BroadcastIndexJoin rast#43: raster, RightSide, LeftSide, Inner, INTERSECTS,  **org.apache.spark.sql.sedona_sql.expressions.raster.RS_Intersects** RS_INTERSECTS(rast#43, geom#53)
               :- Project [ **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTiff**   AS rast#43]
               :  +- Filter isnotnull( **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTiff**  )
               :     +- FileScan binaryFile [content#38] Batched: false, DataFilters: [isnotnull( **org.apache.spa